<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-DataWrangling/blob/main/Session14_BigData_Grundlagen_Uebungen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 14 — Big Data 1: PySpark Übungen

---

## Ziele dieser Übung

Nach diesem Notebook könnt ihr:

- eine SparkSession in Colab aufsetzen und CSV-Daten als Spark-DataFrame einlesen
- Aggregationen mit `groupBy` / `agg` durchführen und den Physical Plan mit `.explain()` interpretieren
- Window-Funktionen für analytische Fragen wie *Top N pro Gruppe* einsetzen
- die Laufzeit von PySpark und pandas auf einem realistischen Datensatz vergleichen und einordnen

## Aufbau

| Teil | Inhalt |
|------|--------|
| 0    | Setup (PySpark Installation, SparkSession, Datensatz) |
| 1    | Aufgabe 1 — Erste Aggregation (★☆☆) |
| 2    | Aufgabe 2 — Top-Kategorie pro Kanton & Monat (★★☆) |
| 3    | Aufgabe 3 — pandas vs. PySpark Benchmark (★★★) |
| 4    | Wrap-up |

> Jede Aufgabe hat zuerst eine **leere Zelle für eure Lösung** und danach eine ausklappbare **Musterlösung**. Erst selber versuchen, dann vergleichen.

---

## Hinweis zur Umgebung

Dieses Notebook ist für **Google Colab** optimiert (kostenloser PySpark-Local-Mode mit mehreren Threads). Es läuft genauso in einem lokalen Jupyter, wenn ihr `pyspark` installiert habt. Im Local-Mode sind die **Konzepte und die API identisch** zu einem echten Cluster — nur die Skalierung fehlt.


---

## 0 · Setup

### 0.1 PySpark installieren

In Colab müsst ihr PySpark einmal pro Session installieren. Lokal könnt ihr die Zelle überspringen, wenn ihr `pyspark` bereits installiert habt.

In [1]:
!pip install pyspark==3.5.1 -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 14.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


### 0.2 SparkSession starten

Die `SparkSession` ist euer Einstiegspunkt zu Spark. Sie hält die Verbindung zum (lokalen oder Cluster-) Spark-Driver.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("Session14_BigData_Grundlagen")
         .config("spark.sql.shuffle.partitions", "8")   # in Colab reichen wenige Partitionen
         .getOrCreate())

# Weniger gesprächiger Log-Output
spark.sparkContext.setLogLevel("WARN")
spark


### 0.3 Synthetischen Datensatz erzeugen

Wir generieren einen Datensatz, der sich an Schweizer Detailhandelsdaten orientiert: ca. **2 Mio. Kassen-Transaktionen** aus dem Jahr 2025, verteilt auf 5 Kantone und 4 Produkt-Kategorien.

> **Hinweis:** Wir erzeugen die Daten *direkt in Spark* — das ist schon ein erster verteilter Job. Bei einem echten Cluster würden diese 2 Mio. Zeilen partitioniert über die Worker erzeugt.

In [3]:
from pyspark.sql.functions import rand, when, col, dayofmonth, expr

N = 2_000_000  # 2 Millionen Transaktionen

# Range-DataFrame erzeugt N Zeilen mit einer id-Spalte — verteilt!
df = spark.range(0, N).withColumnRenamed("id", "transaktion_id")

# Datum: zufälliger Tag im Jahr 2025
df = df.withColumn(
    "datum",
    F.date_add(F.to_date(F.lit("2025-01-01")), (rand(seed=42) * 364).cast("int"))
)

# Kanton (gewichtet realitätsnah: ZH am häufigsten)
df = df.withColumn("r1", rand(seed=1))
df = df.withColumn("kanton",
    when(col("r1") < 0.40, "ZH")
    .when(col("r1") < 0.65, "BE")
    .when(col("r1") < 0.80, "AG")
    .when(col("r1") < 0.92, "BS")
    .otherwise("SO")
)

# Kategorie
df = df.withColumn("r2", rand(seed=2))
df = df.withColumn("kategorie",
    when(col("r2") < 0.45, "Food")
    .when(col("r2") < 0.75, "Haushalt")
    .when(col("r2") < 0.90, "Bekleidung")
    .otherwise("Elektronik")
)

# Betrag (CHF): kategorieabhängig
df = df.withColumn("betrag",
    F.round(
        when(col("kategorie") == "Food",       rand(seed=3) * 80 + 5)
        .when(col("kategorie") == "Haushalt",  rand(seed=4) * 60 + 10)
        .when(col("kategorie") == "Bekleidung", rand(seed=5) * 200 + 20)
        .otherwise(rand(seed=6) * 800 + 50),
        2
    )
)

df = df.drop("r1", "r2")
df.cache()                  # Daten im RAM halten, sonst wird bei jedem Action neu generiert
df.count()                  # Aktion, damit der Cache befüllt wird
print(f"Erzeugte Zeilen: {df.count():,}".replace(",", "'"))


Erzeugte Zeilen: 2'000'000


### 0.4 Erste Sichtprüfung

`.show()` ist eine Action — hier passiert die Berechnung.

In [4]:
df.show(8, truncate=False)
df.printSchema()

+--------------+----------+------+----------+------+
|transaktion_id|datum     |kanton|kategorie |betrag|
+--------------+----------+------+----------+------+
|0             |2025-08-14|BE    |Haushalt  |67.19 |
|1             |2025-07-05|BE    |Food      |25.59 |
|2             |2025-10-31|ZH    |Haushalt  |13.97 |
|3             |2025-04-06|ZH    |Haushalt  |18.67 |
|4             |2025-09-01|BS    |Bekleidung|24.78 |
|5             |2025-07-08|AG    |Food      |58.59 |
|6             |2025-12-30|ZH    |Haushalt  |35.58 |
|7             |2025-01-26|ZH    |Food      |80.33 |
+--------------+----------+------+----------+------+
only showing top 8 rows

root
 |-- transaktion_id: long (nullable = false)
 |-- datum: date (nullable = true)
 |-- kanton: string (nullable = false)
 |-- kategorie: string (nullable = false)
 |-- betrag: double (nullable = true)



---

## 1 · Aufgabe 1 — Erste Aggregation ★☆☆

**Ziel:** Berechne den **Gesamtumsatz pro Kanton** und sortiere absteigend. Schau dir anschliessend mit `.explain()` an, wie Spark den Job plant.

**Erwartete Funktionen:** `groupBy`, `agg`, `F.sum`, `orderBy`, `show`, `explain`

**Tipps:**

- Mit `df.groupBy("kanton").agg(F.sum("betrag").alias("umsatz_chf"))` startet ihr.
- Mit `.orderBy(F.col("umsatz_chf").desc())` sortiert ihr absteigend.
- `df.explain()` zeigt den *Physical Plan* — lest ihn **von unten nach oben**.

**Eure Lösung:**

In [5]:
# TODO Aufgabe 1: Gesamtumsatz pro Kanton, absteigend sortiert



<details>
<summary>📘 <b>Musterlösung anzeigen</b></summary>


In [14]:
# Musterlösung Aufgabe 1
umsatz_pro_kanton = (df
    .groupBy("kanton")
    .agg(F.sum("betrag").alias("umsatz_chf"),
         F.count("*").alias("anzahl_transaktionen"))
    .orderBy(F.col("umsatz_chf").desc())
)

umsatz_pro_kanton.show()

# Physical Plan inspizieren
print("=" * 70)
print("Physical Plan:")
umsatz_pro_kanton.explain()


+------+--------------------+--------------------+
|kanton|          umsatz_chf|anzahl_transaktionen|
+------+--------------------+--------------------+
|    ZH| 7.632745592000057E7|              799545|
|    BE| 4.786541782999943E7|              501041|
|    AG|2.8672658280000106E7|              299918|
|    BS| 2.282491280999995E7|              239877|
|    SO|1.5129827880000092E7|              159619|
+------+--------------------+--------------------+

Physical Plan:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [umsatz_chf#1355 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(umsatz_chf#1355 DESC NULLS LAST, 8), ENSURE_REQUIREMENTS, [plan_id=831]
      +- HashAggregate(keys=[kanton#11], functions=[sum(betrag#29), count(1)])
         +- Exchange hashpartitioning(kanton#11, 8), ENSURE_REQUIREMENTS, [plan_id=828]
            +- HashAggregate(keys=[kanton#11], functions=[partial_sum(betrag#29), partial_count(1)])
               +- InMemoryTableScan [kanton#1

**Was sagt der Physical Plan?**

Lest die Stages von **unten nach oben**:

1. **InMemoryTableScan** / **FileScan** — Spark liest die Quelle (bei uns aus dem Cache).
2. **HashAggregate (partial_sum)** — jeder Worker rechnet eine **Teilsumme** pro Kanton.
3. **Exchange hashpartitioning(kanton, …)** — *Shuffle*: alle Teilsummen desselben Kantons werden zusammengebracht.
4. **HashAggregate (sum)** — finale Summe pro Kanton.
5. **Sort / TakeOrderedAndProject** — Sortierung am Ende.

Spark wendet hier **partielle Aggregation** an: das reduziert die Datenmenge *vor* dem teuren Shuffle drastisch. Das ist eine der wichtigsten Optimierungen.

</details>


---

## 2 · Aufgabe 2 — Top-Kategorie pro Kanton und Monat ★★☆

**Ziel:** Für jeden Kanton und jeden Monat des Jahres 2025: Welche **Produktkategorie** hatte den **höchsten Umsatz**?

Das ist eine klassische *Top-N-per-Group*-Frage — der Standard-Lösungsweg sind **Window-Funktionen**.

**Erwartete Funktionen:** `withColumn`, `F.month`, `groupBy`, `F.sum`, `Window.partitionBy`, `F.row_number`, `filter`

**Vorgehen (Tipp):**

1. Eine Spalte `monat` aus `datum` extrahieren mit `F.month("datum")`.
2. Mit `groupBy("kanton", "monat", "kategorie").agg(F.sum("betrag"))` den Umsatz aggregieren.
3. Ein `Window` über `partitionBy("kanton", "monat")` und `orderBy(F.col("umsatz").desc())` definieren.
4. `F.row_number().over(w)` als Rang hinzufügen.
5. Nach `rang == 1` filtern.

**Eure Lösung:**

In [7]:
# TODO Aufgabe 2: Top-Kategorie pro Kanton und Monat



<details>
<summary>📘 <b>Musterlösung anzeigen</b></summary>


In [8]:
# Musterlösung Aufgabe 2

# Schritt 1+2: Aggregat pro Kanton/Monat/Kategorie
monatsumsatz = (df
    .withColumn("monat", F.month("datum"))
    .groupBy("kanton", "monat", "kategorie")
    .agg(F.sum("betrag").alias("umsatz"))
)

# Schritt 3+4: Window-Funktion zum Ranking
w = Window.partitionBy("kanton", "monat").orderBy(F.col("umsatz").desc())

ranked = monatsumsatz.withColumn("rang", F.row_number().over(w))

# Schritt 5: Nur Top-1 pro (Kanton, Monat)
top_pro_monat = (ranked
    .filter(F.col("rang") == 1)
    .select("kanton", "monat", "kategorie", "umsatz")
    .orderBy("kanton", "monat")
)

top_pro_monat.show(60)


+------+-----+----------+------------------+
|kanton|monat| kategorie|            umsatz|
+------+-----+----------+------------------+
|    AG|    1|Elektronik|1178393.5800000026|
|    AG|    2|Elektronik|1068007.6800000006|
|    AG|    3|Elektronik|1168010.2099999995|
|    AG|    4|Elektronik|1120591.4400000006|
|    AG|    5|Elektronik|        1144738.11|
|    AG|    6|Elektronik|1105671.7599999993|
|    AG|    7|Elektronik|1198572.1099999996|
|    AG|    8|Elektronik|        1134164.03|
|    AG|    9|Elektronik|        1117116.69|
|    AG|   10|Elektronik|1154471.0499999998|
|    AG|   11|Elektronik|1118287.2400000007|
|    AG|   12|Elektronik|1126434.1999999993|
|    BE|    1|Elektronik|1914946.8999999994|
|    BE|    2|Elektronik| 1755243.029999999|
|    BE|    3|Elektronik| 1913586.269999999|
|    BE|    4|Elektronik|1893656.1700000013|
|    BE|    5|Elektronik|1923925.7300000032|
|    BE|    6|Elektronik|1876196.3900000001|
|    BE|    7|Elektronik|1980921.2299999974|
|    BE|  

**Reflexionsfragen:**

- Warum braucht es zwei Phasen (erst aggregieren, dann Window)? *(Das `Window` kann nicht direkt auf Roh-Transaktionen angewendet werden, weil wir pro Gruppe **summieren** wollen.)*
- Welche Kategorie dominiert in jedem Kanton? Gibt es Überraschungen? *(Bei unseren synthetischen Daten dominiert Elektronik wegen der hohen Beträge.)*

</details>


---

## 3 · Aufgabe 3 — pandas vs. PySpark Benchmark ★★★

**Ziel:** Vergleicht für **dieselbe Analyse** die Laufzeit von **pandas** und **PySpark** auf unseren 2 Mio. Zeilen. Reflektiert anschliessend, **wann sich Spark lohnt** und wann nicht.

**Analyse:** *Durchschnittlicher Transaktionsbetrag pro Kategorie und Kanton.*

**Vorgehen:**

1. Wandelt das Spark-DataFrame mit `df.toPandas()` in ein pandas-DataFrame um (Achtung: das materialisiert alles im Speicher des Drivers!).
2. Misst die Laufzeit der Analyse in pandas mit `time.perf_counter()`.
3. Misst die Laufzeit derselben Analyse in PySpark.
4. Vergleicht die beiden Resultate inhaltlich (gleich?) und zeitlich (welches ist schneller, um wie viel?).


**Eure Lösung:**

In [9]:
# TODO Aufgabe 3: pandas vs. PySpark Benchmark



<details>
<summary>📘 <b>Musterlösung anzeigen</b></summary>


In [10]:
# Musterlösung Aufgabe 3
import time

# === 1. Spark → pandas konvertieren ===
t0 = time.perf_counter()
pdf = df.toPandas()
t_convert = time.perf_counter() - t0
print(f"Spark → pandas Konvertierung: {t_convert:.2f} s   ({len(pdf):,} Zeilen)".replace(",", "'"))

# === 2. Pandas-Analyse ===
t0 = time.perf_counter()
pandas_result = (pdf
    .groupby(["kategorie", "kanton"])["betrag"]
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={"betrag": "avg_betrag_chf"})
    .sort_values(["kategorie", "kanton"])
)
t_pandas = time.perf_counter() - t0
print(f"pandas-Analyse: {t_pandas:.3f} s")


Spark → pandas Konvertierung: 30.98 s   (2'000'000 Zeilen)
pandas-Analyse: 0.741 s


In [11]:
# === 3. PySpark-Analyse ===
t0 = time.perf_counter()
spark_result = (df
    .groupBy("kategorie", "kanton")
    .agg(F.round(F.avg("betrag"), 2).alias("avg_betrag_chf"))
    .orderBy("kategorie", "kanton")
)
spark_result_collected = spark_result.collect()   # Action erzwingen
t_spark = time.perf_counter() - t0
print(f"PySpark-Analyse: {t_spark:.3f} s")


PySpark-Analyse: 2.921 s


In [12]:
# === 4. Ergebnisse vergleichen ===
print("pandas-Resultat:")
print(pandas_result.to_string(index=False))
print()
print("PySpark-Resultat:")
spark_result.show(20, truncate=False)

print(f"\nVerhältnis pandas / PySpark: {t_pandas / t_spark:.2f}x")


pandas-Resultat:
 kategorie kanton  avg_betrag_chf
Bekleidung     AG          119.68
Bekleidung     BE          119.88
Bekleidung     BS          120.41
Bekleidung     SO          119.87
Bekleidung     ZH          120.37
Elektronik     AG          452.24
Elektronik     BE          449.82
Elektronik     BS          449.44
Elektronik     SO          446.77
Elektronik     ZH          450.83
      Food     AG           44.90
      Food     BE           45.03
      Food     BS           45.06
      Food     SO           45.10
      Food     ZH           45.02
  Haushalt     AG           39.99
  Haushalt     BE           39.94
  Haushalt     BS           39.96
  Haushalt     SO           40.13
  Haushalt     ZH           39.99

PySpark-Resultat:
+----------+------+--------------+
|kategorie |kanton|avg_betrag_chf|
+----------+------+--------------+
|Bekleidung|AG    |119.68        |
|Bekleidung|BE    |119.88        |
|Bekleidung|BS    |120.41        |
|Bekleidung|SO    |119.87        |
|Bekl

**Erwartete Beobachtungen:**

- Auf einem kleinen Datensatz (2 Mio. Zeilen, alles im RAM) ist **pandas oft schneller**, weil Spark Startup-Overhead und Serialisierung hat.
- Spark glänzt erst, wenn die Daten **nicht mehr in den Driver-RAM passen** oder wenn echte Parallelisierung über mehrere Knoten möglich ist.
- Die *Konvertierung* `toPandas()` ist der teuerste Schritt — sie kollabiert die Verteilung.

**Take-away:** Spark ist ein **Werkzeug, kein Reflex**. Erst wenn pandas/Polars/DuckDB an die Grenze stossen, lohnt sich der Wechsel.

</details>


---

## 4 · Bonus — Catalyst Optimizer beobachten

Spark ist nicht magisch — der **Catalyst Optimizer** schreibt eure Queries automatisch um. Probiert es aus:

1. Schreibt eine Query mit `filter` **nach** dem `groupBy` (logisch unsinnig).
2. Schaut mit `.explain()`, wo der Filter im Physical Plan landet.

In [13]:
# Filter nach groupBy formuliert
abfrage = (df
    .groupBy("kanton", "kategorie")
    .agg(F.sum("betrag").alias("umsatz"))
    .filter(F.col("kanton") == "ZH")     # erst hier filtern wir
)

abfrage.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[kanton#11, kategorie#22], functions=[sum(betrag#29)])
   +- Exchange hashpartitioning(kanton#11, kategorie#22, 8), ENSURE_REQUIREMENTS, [plan_id=726]
      +- HashAggregate(keys=[kanton#11, kategorie#22], functions=[partial_sum(betrag#29)])
         +- Filter (kanton#11 = ZH)
            +- InMemoryTableScan [kanton#11, kategorie#22, betrag#29], [(kanton#11 = ZH)]
                  +- InMemoryRelation [transaktion_id#2L, datum#4, kanton#11, kategorie#22, betrag#29], StorageLevel(disk, memory, deserialized, 1 replicas)
                        +- *(1) Project [transaktion_id#2L, datum#4, kanton#11, kategorie#22, round(CASE WHEN (kategorie#22 = Food) THEN ((rand(3) * 80.0) + 5.0) WHEN (kategorie#22 = Haushalt) THEN ((rand(4) * 60.0) + 10.0) WHEN (kategorie#22 = Bekleidung) THEN ((rand(5) * 200.0) + 20.0) ELSE ((rand(6) * 800.0) + 50.0) END, 2) AS betrag#29]
                           +- *(1) Project [transaktio

**Beobachtung:** Catalyst schiebt den Filter **vor** das `groupBy` (*Predicate Pushdown*). Im Physical Plan steht der Filter unten, vor der Aggregation. Das spart enorm viel Arbeit, weil weniger Daten geshuffelt werden müssen.

Diese Optimierungen sind das Herzstück von Sparks Geschwindigkeit.

---

## 5 · Wrap-up

Wir haben heute:

- den Unterschied zwischen **Transformations** (lazy) und **Actions** (eager) gesehen
- mit dem **DataFrame-API** gearbeitet (sehr ähnlich zu pandas)
- den **Physical Plan** mit `.explain()` inspiziert und partielle Aggregation entdeckt
- eine **Window-Funktion** für Top-N-per-Group eingesetzt
- pandas und PySpark **fair verglichen** und gemerkt: für kleine Daten ist pandas oft schneller

### Vor Session 15 (Big Data 2 — unstrukturierte Daten)

- Probiert die obigen Aufgaben mit `N = 200_000` (10x kleiner) und `N = 20_000_000` (10x grösser, ggf. Colab Pro nötig). Wie verschieben sich die Verhältnisse pandas ↔ PySpark?
- Schaut in die Spark UI unter `http://localhost:4040` (in Colab via `pyngrok`). Wie viele Stages erzeugt Aufgabe 2?

### Quellen und weiterführend

- Spark-Dokumentation: <https://spark.apache.org/docs/latest/sql-programming-guide.html>
- Databricks Learning Academy (kostenlos): <https://www.databricks.com/learn>